In [ ]:
import subprocess

def call_tree_diagram(target, depth=20, diagram='sequence',
                      show_conditions=False, params=None, prune=False,
                      label_suffix=''):
    """Render a call-tree diagram for `target` as a quarto mermaid/txt block.

    show_conditions : annotate every branch with the guard condition that selects it.
    params          : a "k=v,..." string; resolve which path those concrete values
                      take (edges marked ✓ taken / ✗ not-taken / ? undecidable —
                      kept and marked, never guessed).
    prune           : with `params`, drop the provably not-taken branches.
    label_suffix    : disambiguate the quarto figure label when several diagrams
                      share the same target.
    """
    assert diagram in ['sequence', 'flow', 'ascii']
    project_root = '../../..'
    cmd = ['python', f'{project_root}/scripts/call_tree_analyzer.py',
           f'{project_root}/src/phasic', '--quiet', '--callable', target]
    if diagram != 'ascii':
        cmd += ['--diagram', diagram]
    if show_conditions:
        cmd += ['--show-conditions']
    if params:
        cmd += ['--params', params]   # single argument, no shell quoting needed
    if prune:
        cmd += ['--prune']
    cmd += ['-d', str(depth)]
    output = subprocess.check_output(cmd).decode()
    if diagram == 'ascii':
        print(f"```{{txt}}\n{output}\n```")
    else:
        label = 'fig-diagram-' + target.replace('.', '-')
        if label_suffix:
            label += '-' + label_suffix
        print(f"```{{mermaid}}\n%%| echo: false\n%%| label: {label}\n{output}\n```")

In [ ]:
#| echo: false
#| output: asis
call_tree_diagram("Graph.svgd", diagram='sequence')

In [ ]:
#| echo: false
#| output: asis
call_tree_diagram("Graph", diagram='ascii')

In [ ]:
#| echo: false
#| output: asis
call_tree_diagram("Graph.svgd", diagram='ascii')

In [ ]:
#| echo: false
#| output: asis
call_tree_diagram("Graph.svgd", diagram='flow')

See @fig-diagram-Graph-svgd for the diagram...

In [ ]:
#| echo: false
#| output: asis
call_tree_diagram("Graph.svgd", diagram='ascii', depth=100)

## Parameter-dependent paths

The diagrams above enumerate every call `Graph.svgd` *can* make. Two flags turn
that enumeration into a path view:

- `--show-conditions` labels each edge with the branch condition that selects it
  (@fig-diagram-Graph-svgd-conditions).
- `--params "k=v,…"` resolves the edges for concrete inputs — ✓ taken,
  ✗ not-taken (dotted), ? undecidable — and `--prune` keeps only the branches
  actually taken (@fig-diagram-Graph-svgd-params).

Branches gated by runtime values, instance attributes, or `**kwargs` forwarding
cannot be decided statically and stay `?` (kept and marked, never guessed).

In [ ]:
#| echo: false
#| output: asis
call_tree_diagram("Graph.svgd", diagram='flow', show_conditions=True,
                  depth=2, label_suffix='conditions')

In [ ]:
#| echo: false
#| output: asis
call_tree_diagram(
    "Graph.svgd", diagram='flow',
    params="rewards=None,callback=None,weight_formula=None,"
           "epoch_starts=None,joint_index=False,discrete=False",
    prune=True, depth=2, label_suffix='params')